In [1]:
import pandas as pd
import numpy as np
import json

DATA_PATH = "../data/processed/bank_clean.csv"
KPI_PATH = "../data/processed/kpis.csv"
SEG_PATH = "../data/processed/segment_kpis.csv"

pd.set_option("display.max_columns", None)

df = pd.read_csv(DATA_PATH)


In [3]:
orders = {
    "AgeBand": ["18-29", "30-39", "40-49", "50-59", "60+"],
    "TenureBand": ["0-2", "3-5", "6-8", "9-10"],
    "CreditBand": ["Poor", "Fair", "Good", "Very Good+"],
    "BalanceBand": ["Zero", "Low", "Mid", "High"],
    "ProductGroup": ["1", "2", "3+"],
}
for col, order in orders.items():
    df[col] = df[col].astype(str)
    df[col] = pd.Categorical(df[col], categories=order, ordered=True)
import pandas as pd
import numpy as np
import json

DATA_PATH = "../data/processed/bank_clean.csv"
KPI_PATH = "../data/processed/kpis.json"
SEG_PATH = "../data/processed/segment_kpis.csv"

pd.set_option("display.max_columns", None)

df = pd.read_csv(DATA_PATH)

# Reloading a CSV drops the ordered categories, so re-apply them
orders = {
    "AgeBand": ["18-29", "30-39", "40-49", "50-59", "60+"],
    "TenureBand": ["0-2", "3-5", "6-8", "9-10"],
    "CreditBand": ["Poor", "Fair", "Good", "Very Good+"],
    "BalanceBand": ["Zero", "Low", "Mid", "High"],
    "ProductGroup": ["1", "2", "3+"],
}
for col, order in orders.items():
    df[col] = df[col].astype(str)
    df[col] = pd.Categorical(df[col], categories=order, ordered=True)

df["HighValue"] = (df["BalanceBand"] == "High").astype(int)
df["Priority"] = (
    (df["Balance"] >= 100_000)
    & (df["IsActiveMember"] == 0)
    & (df["ProductGroup"] == "1")
    & (df["AgeBand"].isin(["40-49", "50-59"]))
).astype(int)

# Sanity checks: shape (10000, 19), HighValue 2128, Priority 618
print(df.shape)
print(df["HighValue"].sum(), df["Priority"].sum())
print(df[orders.keys()].isna().sum().sum())  # must be 0

(10000, 19)
2128 618
0


In [5]:
total_churners = df["Exited"].sum()
total_balance_leaving = df.loc[df["Exited"] == 1, "Balance"].sum()

def kpi(sub):
    return{
        "customers": int(len(sub)),
        "churned": int(sub["Exited"].sum()),
        "churn_rate": round(float(sub["Exited"].mean()),4),
        "balance_total": round(float(sub["Balance"].sum()),2),
        "Balance_leaving": round(float(sub.loc[sub["Exited"] == 1, "Balance"].sum()), 2),
        "share_of_all_balance_leaving": round(
            float(sub.loc[sub["Exited"] == 1, "Balance"].sum() / total_balance_leaving),4
        ),
    }
kpis = {
    "overall":kpi(df),
    "high_value": kpi(df[df["HighValue"] == 1]),
    "priority": kpi(df[df["Priority"] == 1]),
}

kpi_table = pd.DataFrame(kpis).T
print(kpi_table)

with open(KPI_PATH, "w") as f:
    json.dump(kpis, f, indent=2)

            customers  churned  churn_rate  balance_total  Balance_leaving  \
overall       10000.0   2037.0      0.2037   7.648589e+08     1.855881e+08   
high_value     2128.0    493.0      0.2317   3.243576e+08     7.579534e+07   
priority        618.0    353.0      0.5712   8.187845e+07     4.657205e+07   

            share_of_all_balance_leaving  
overall                           1.0000  
high_value                        0.4084  
priority                          0.2509  


In [12]:
overall_rate = df["Exited"].mean()
cols =  ["Geography", "AgeBand", "ProductGroup", "IsActiveMember", "Gender", "BalanceBand"]

frames = []
for col in cols:
    g = df.groupby(col, observed=True).agg(
        customers=("Exited","size"),
        churned=("Exited","sum"),
        churn_rate=("Exited","mean"),
    ).reset_index().rename(columns={col: "segment"})
    g.insert(0, "dimension", col)
    g["share_of_churners"] = g["churned"] / total_churners
    g["lift"] = g["churn_rate"] / overall_rate
    frames.append(g)

seg = pd.concat(frames, ignore_index=True)
seg[["churn_rate","share_of_churners","lift"]] = seg[["churn_rate","share_of_churners","lift"]].round(4)
print(seg)

seg.to_csv(SEG_PATH, index=False)

         dimension  segment  customers  churned  churn_rate  \
0        Geography   France       5014      810      0.1615   
1        Geography  Germany       2509      814      0.3244   
2        Geography    Spain       2477      413      0.1667   
3          AgeBand    18-29       1641      124      0.0756   
4          AgeBand    30-39       4346      473      0.1088   
5          AgeBand    40-49       2618      806      0.3079   
6          AgeBand    50-59        869      487      0.5604   
7          AgeBand      60+        526      147      0.2795   
8     ProductGroup        1       5084     1409      0.2771   
9     ProductGroup        2       4590      348      0.0758   
10    ProductGroup       3+        326      280      0.8589   
11  IsActiveMember        0       4849     1302      0.2685   
12  IsActiveMember        1       5151      735      0.1427   
13          Gender   Female       4543     1139      0.2507   
14          Gender     Male       5457      898      0.